# 04 Pretrained Whisper-small on our training clips (raw ASR output for Q2)

Same model, same eval_norm, same token-cap rule as notebook 03, so the outputs are comparable.
Kaggle: Accelerator = GPU, Internet = On, input = `echoscript-clean`.

In [ ]:
!pip install -q jiwer


In [ ]:
import os, glob, re, time, unicodedata
import numpy as np, pandas as pd, soundfile as sf, torch, jiwer
from transformers import WhisperProcessor, WhisperForConditionalGeneration

ROOT = os.path.dirname(glob.glob("/kaggle/input/**/manifest.csv", recursive=True)[0])
print("ROOT:", ROOT, "| GPU:", torch.cuda.get_device_name(0))

def eval_norm(t):
    t = unicodedata.normalize("NFC", t).replace("_", " ")
    t = re.sub(r"[,.।॥!?:;\-\"'“”‘’()]", " ", t)
    return re.sub(r"\s+", " ", t).strip().lower()

device = "cuda"
proc = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device).half().eval()

def decode_batch(arrays, max_new_tokens=None):
    feats = proc.feature_extractor(arrays, sampling_rate=16000,
                                   return_tensors="pt").input_features.to(device).half()
    kw = {} if max_new_tokens is None else {"max_new_tokens": int(min(max_new_tokens, 440))}
    with torch.no_grad():
        ids = model.generate(feats, language="hi", task="transcribe", **kw)
    return proc.batch_decode(ids, skip_special_tokens=True)

# same cap rule as notebook 03 (computed over the whole manifest, as before)
man = pd.read_csv(f"{ROOT}/manifest.csv")
man["tokens"] = [len(proc.tokenizer(t, add_special_tokens=False).input_ids) for t in man.text]
ok = man[man.duration >= 1.0]
tps99 = float(np.percentile(ok.tokens / ok.duration, 99))
print("tokens/sec 99th percentile:", round(tps99, 1), "(notebook 03 printed 20.4)")
def cap_for(d): return max(10, 1.5 * tps99 * d + 5)


In [ ]:
train = man[man.split == "train"].sort_values("duration").reset_index(drop=True)
print("train clips:", len(train))
t0 = time.time()
hyp = []
for i in range(0, len(train), 16):
    chunk = train.iloc[i:i+16]
    arrs = [sf.read(f"{ROOT}/clips/{c}")[0] for c in chunk.clip]
    hyp += decode_batch(arrs, max_new_tokens=cap_for(chunk.duration.max()))
    if (i // 16) % 50 == 0:
        print(i, "clips done,", round((time.time() - t0) / 60, 1), "min")
print("train decode time (min):", round((time.time() - t0) / 60, 1))

train["hyp_raw"] = hyp
train["ref_n"] = [eval_norm(x) for x in train.text]
train["hyp_n"] = [eval_norm(x) for x in train.hyp_raw]
train.to_csv("/kaggle/working/train_baseline_predictions.csv", index=False, encoding="utf-8-sig")

print("WER on train clips (eval_norm, capped):", round(jiwer.wer(list(train.ref_n), list(train.hyp_n)), 4))
runaway = (train.hyp_n.str.split().str.len() > 3 * train.ref_n.str.split().str.len() + 5)
print("runaway clips:", int(runaway.sum()), "of", len(train))
